# Day 6: The record check, and the results with and without the countries that fail it

Day 1 leaves seven EU countries out of the sample because their records fail a consistency check. This notebook gives the evidence in full and shows what the main results look like if those countries are kept in. In the public data file (GESIS ZA8869, version 1.0.0) the answers from **eight countries** do not hang together the way answers from real firms must.

**The test.** A firm with more employees has a higher turnover. In any genuine business survey the two are strongly related. If they are unrelated inside a country, then the turnover on a row does not belong to the firm described on that row.

| Part | Question |
|---|---|
| A | In which countries are employees and turnover unrelated? (raw GESIS file, all 38 countries) |
| B | Is it only that pair, or do answers in general fail to hang together there? |
| C | Can the models predict anything inside those countries? |
| D | How do the main results differ with and without those countries? |

**How to run it:** *Runtime > Run all*, then upload three files together: `ZA8869_v1-0-0.sav` (the GESIS file), `sme_energy_27countries.csv` (from Day 1) and `country_indicators.csv`. It takes roughly 10 to 20 minutes.

In [ ]:
!pip -q install pyreadstat

In [ ]:
import os, time, warnings
import pyreadstat
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score, confusion_matrix, precision_recall_fscore_support

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 100)
SEED = 42

NEEDED = ["ZA8869_v1-0-0.sav", "sme_energy_27countries.csv", "country_indicators.csv"]
if not all(os.path.exists(f) for f in NEEDED):
    try:
        from google.colab import files
        print("Choose all THREE files:", NEEDED); files.upload()
    except ImportError:
        raise FileNotFoundError(f"Put {NEEDED} in the same folder as this notebook.")

df = pd.read_csv("sme_energy_27countries.csv")          # all 27 countries; the column records_ok marks the 20 used in the paper
indicators = pd.read_csv("country_indicators.csv").set_index("country")
CLASS_NAMES = ["No renewable", "Purchased", "On-site"]
y = df["pathway_code"]
df["log_employees"] = np.log1p(df["employees"])
df["log_firm_age"] = np.log1p(df["firm_age_years"].clip(lower=0))

FIRM = ["nace_section", "size", "log_employees", "log_firm_age", "age_band", "turnover_band",
        "turnover_trend", "employment_trend", "hiring_difficulty", "sells"]
BEHAVIOUR = ([c for c in df.columns if c.startswith(("act_", "sup_", "ext_", "bar_", "help_"))]
             + ["n_other_actions", "n_barriers", "asked_followups", "investment", "climate_strategy",
                "green_products", "green_jobs", "recycled_materials", "lifespan_policy"])
SETS = {"A1 Firm": FIRM, "A2 Firm + country": FIRM + ["country"], "B All": FIRM + ["country"] + BEHAVIOUR}
NUM_WITH_GAPS = ["log_employees", "log_firm_age"]

def make_pre(columns):
    cat = [c for c in columns if not pd.api.types.is_numeric_dtype(df[c])]
    gaps = [c for c in columns if c in NUM_WITH_GAPS]
    num = [c for c in columns if pd.api.types.is_numeric_dtype(df[c]) and c not in NUM_WITH_GAPS]
    steps = [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat),
             ("gaps", Pipeline([("fill", SimpleImputer(strategy="median", add_indicator=True)), ("scale", StandardScaler())]), gaps)]
    if num: steps.append(("num", StandardScaler(), num))
    return ColumnTransformer(steps)

# The settings selected by cross-validation on Day 2. They are held fixed in every check below.
# (See the "Cross-validation" sheet of Day2_results.xlsx.)
LOGIT_C = {"A1 Firm": 1.0, "A2 Firm + country": 0.1, "B All": 0.1}
RF_SET = {"A1 Firm": dict(min_samples_leaf=10, max_features="sqrt"), "A2 Firm + country": dict(min_samples_leaf=10, max_features="sqrt"),
          "B All": dict(min_samples_leaf=3, max_features=0.3)}
XGB_SET = {"A1 Firm": dict(max_depth=3, learning_rate=0.03, n_estimators=200, colsample_bytree=0.9, min_child_weight=5, subsample=0.8),
           "A2 Firm + country": dict(max_depth=3, learning_rate=0.03, n_estimators=400, colsample_bytree=0.9, min_child_weight=5, subsample=0.8),
           "B All": dict(max_depth=3, learning_rate=0.1, n_estimators=200, colsample_bytree=0.6, min_child_weight=1, subsample=0.8)}
def new_logit(set_name, class_weight="balanced"):
    return LogisticRegression(max_iter=3000, C=LOGIT_C[set_name], class_weight=class_weight)
def new_rf(set_name):
    return RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED, **RF_SET[set_name])
def new_xgb(set_name="B All"):
    return xgb.XGBClassifier(objective="multi:softprob", tree_method="hist", n_jobs=2, random_state=SEED, verbosity=0, **XGB_SET[set_name])
def auc(y_true, proba, w=None):
    return roc_auc_score(y_true, proba, multi_class="ovr", sample_weight=w)

def theme_of(p):
    if p == "country": return "Country"
    if p in ("size", "log_employees"): return "Firm size"
    if p in ("turnover_band", "turnover_trend", "employment_trend"): return "Turnover and growth"
    if p in ("nace_section", "sells"): return "Sector and offer"
    if p in ("log_firm_age", "age_band"): return "Firm age"
    if p == "hiring_difficulty": return "Hiring difficulty"
    if p.startswith("act_") or p in ("n_other_actions", "asked_followups"): return "Other green actions"
    if p == "investment": return "Investment level"
    if p.startswith(("sup_", "ext_")): return "Support used"
    if p.startswith("bar_") or p == "n_barriers": return "Barriers met"
    if p.startswith("help_"): return "Support wanted"
    if p == "climate_strategy": return "Climate strategy"
    return "Green products, jobs, materials"

def shap_by_predictor(model, pre, X_enc, predictors):
    # Exact TreeSHAP on the model's raw class scores, summed back from one-hot columns to predictors.
    contrib = model.get_booster().predict(xgb.DMatrix(X_enc), pred_contribs=True)[:, :, :-1]
    out = np.zeros((X_enc.shape[0], 3, len(predictors)))
    for j, name in enumerate(pre.get_feature_names_out()):
        out[:, :, predictors.index(owner(name, predictors))] += contrib[:, :, j]
    return out
def theme_importance(shap, predictors, w=None):
    themes = sorted(set(theme_of(p) for p in predictors))
    st = np.zeros((shap.shape[0], 3, len(themes)))
    for j, p in enumerate(predictors):
        st[:, :, themes.index(theme_of(p))] += shap[:, :, j]
    return pd.DataFrame(np.average(np.abs(st), axis=0, weights=w).T, index=themes, columns=CLASS_NAMES)

def owner(name, predictors):
    # Which original predictor does an encoded column belong to?
    cat = [c for c in predictors if not pd.api.types.is_numeric_dtype(df[c])]
    block, rest = name.split("__", 1)
    return max((c for c in cat if rest.startswith(c + "_")), key=len) if block == "cat" else rest.replace("missingindicator_", "")
def linear_by_predictor(logit, pre, X_enc, centre, predictors):
    # The same quantity for a logistic regression: coefficient x (value - average value), on the model's raw class scores.
    lin = (X_enc - centre)[:, None, :] * logit.coef_[None, :, :]
    out = np.zeros((X_enc.shape[0], 3, len(predictors)))
    for j, name in enumerate(pre.get_feature_names_out()):
        out[:, :, predictors.index(owner(name, predictors))] += lin[:, :, j]
    return out
def balanced_weights(y_part, w_part):
    # Each firm's weight, rescaled so the three pathways carry equal total weight. With w = 1 this is the usual "balanced" class weight.
    w_part = np.asarray(w_part, dtype=float); y_part = np.asarray(y_part)
    class_total = pd.Series(w_part).groupby(y_part).sum()
    sw = w_part * w_part.sum() / (3 * class_total.loc[y_part].values)
    return sw / sw.mean()
WEIGHTINGS = {"No survey weights (main)": None, "Within-country weights": "weight_country", "Population weights": "weight_eu27"}
def survey_w(index, scheme):
    return np.ones(len(index)) if WEIGHTINGS[scheme] is None else df.loc[index, WEIGHTINGS[scheme]].values

def run_split(tr, te, set_name, scheme):
    # Fit logistic regression and XGBoost on one split under one weighting scheme; return scores and theme importances.
    cols = SETS[set_name]; pre = make_pre(cols); Xtr = pre.fit_transform(df.loc[tr, cols]); Xte = pre.transform(df.loc[te, cols])
    wtr, wte = survey_w(tr, scheme), survey_w(te, scheme)
    sw = balanced_weights(y.loc[tr], wtr)
    lg = LogisticRegression(max_iter=3000, C=LOGIT_C[set_name]).fit(Xtr, y.loc[tr], sample_weight=sw)
    xg = new_xgb(set_name).fit(Xtr, y.loc[tr], sample_weight=sw)
    out = {"proba": {"Logistic regression": lg.predict_proba(Xte), "XGBoost": xg.predict_proba(Xte)}}
    out["themes"] = {"XGBoost": theme_importance(shap_by_predictor(xg, pre, Xte, cols), cols, w=wte),
                     "Logistic regression": theme_importance(linear_by_predictor(lg, pre, Xte, np.average(Xtr, axis=0, weights=wtr), cols), cols, w=wte)}
    out["fitted"] = (pre, Xtr, Xte, lg, xg)
    return out

train_idx, test_idx = train_test_split(df.index, test_size=0.20, stratify=y, random_state=SEED)     # the Day 2 split
N_SPLITS = 20
SPLITS = [train_test_split(df.index, test_size=0.20, stratify=y, random_state=1000 + s) for s in range(N_SPLITS)]
print("Firms:", len(df))

## A. Employees and turnover, country by country

This part uses the raw GESIS file, before any of our own preparation, so the result cannot come from our code. For each of the 38 countries it computes the rank correlation between the number of employees and the turnover band.

Two control pairs are included. Size class and number of employees are two answers to the same question, and so are age band and year founded. These should agree everywhere.

In [ ]:
raw, meta = pyreadstat.read_sav("ZA8869_v1-0-0.sav")
r = raw.copy()
r["employees"] = r["scr10a"].where(r["scr10a"] < 99990)          # the top codes mean "don't know"
r["turnover"] = r["scr14"].where(r["scr14"] < 10)
r["year"] = r["scr12a"].where(r["scr12a"] < 9998)
def rho(g, a, b):
    x = g[[a, b]].dropna()
    return np.nan if len(x) < 30 or x[a].nunique() < 2 or x[b].nunique() < 2 else spearmanr(x[a], x[b])[0]
partA = pd.DataFrame({c: {"Firms": len(g), "EU27": "yes" if g["eu27"].iloc[0] == 1 else "no",
                          "Employees vs turnover": rho(g, "employees", "turnover"),
                          "Control: size class vs employees": rho(g, "scr10", "employees"),
                          "Control: age band vs year founded": abs(rho(g, "scr12", "year"))} for c, g in r.groupby("isocntry")}).T
num = ["Employees vs turnover", "Control: size class vs employees", "Control: age band vs year founded"]
partA[num] = partA[num].astype(float); partA["Firms"] = partA["Firms"].astype(int)
partA = partA.sort_values("Employees vs turnover")
AFFECTED = sorted(partA.index[partA["Employees vs turnover"] < 0.2])
AFFECTED_EU = sorted(c for c in AFFECTED if partA.loc[c, "EU27"] == "yes")
print("Countries where employees and turnover are unrelated (correlation below 0.2):", AFFECTED)
print("Of these, in the EU27:", AFFECTED_EU)
print("Lowest correlation among all the other countries: %.2f" % partA.loc[~partA.index.isin(AFFECTED), "Employees vs turnover"].min())
partA.round(2)

**What to look for.** The first column should split cleanly into two groups: a handful of countries near zero, and the rest well above. The two control columns should be high everywhere, which shows that answers *within* one question are intact and it is the link *between* questions that is broken.

## B. Is it only turnover?

If the problem were confined to the turnover question we could drop turnover and carry on. This part checks pairs of answers from different parts of the questionnaire, on the prepared 27-country file (12,832 SMEs), separately for the affected countries and for the other 20.

In [ ]:
assert set(df.loc[df["records_ok"] == 0, "country"]) == set(AFFECTED_EU), "the Day 1 file and this check disagree"
df["affected"] = np.where(df["country"].isin(AFFECTED_EU), "Affected countries", "Other 20 countries")
q13 = raw.set_index("uniqid")["q1.3"]; df["q1_3"] = df["firm_id"].map(q13)       # the item we excluded as leakage
df["uses_renewable"] = (df["pathway"] != "No renewable").astype(int)
df["on_site"] = (df["pathway"] == "On-site").astype(int); df["purchased"] = (df["pathway"] == "Purchased").astype(int)
df["investment_level"] = df["investment"].str[0].where(df["investment"].str[0].isin(list("012345"))).astype(float)
df["has_strategy"] = (df["climate_strategy"] == "Has a strategy").astype(int)
turn_order = {b: i for i, b in enumerate(sorted(x for x in df["turnover_band"].dropna().unique() if x[0].isdigit()))}
df["turnover_level"] = df["turnover_band"].map(turn_order)
PAIRS = [("employees", "turnover_level", "Employees and turnover"),
         ("employees", "investment_level", "Employees and investment in resource efficiency"),
         ("employees", "has_strategy", "Employees and having a climate strategy"),
         ("n_other_actions", "investment_level", "Number of green actions and investment"),
         ("n_other_actions", "has_strategy", "Number of green actions and climate strategy"),
         ("ext_public_funding", "investment_level", "Public funding and investment"),
         ("n_other_actions", "plans_renewable", "Number of green actions and planning renewables"),
         ("n_other_actions", "uses_renewable", "Number of green actions and using renewables"),
         ("investment_level", "on_site", "Investment and on-site generation"),
         ("employees", "on_site", "Employees and on-site generation"),
         ("has_strategy", "purchased", "Climate strategy and purchasing"),
         ("q1_3", "uses_renewable", "Control: 'uses mostly renewable energy' (Q1.3) and using renewables (N1a)")]
partB = pd.DataFrame({g: {label: rho(d, a, b) for a, b, label in PAIRS} for g, d in df.groupby("affected")})[["Other 20 countries", "Affected countries"]]
print("Firms:", df["affected"].value_counts().to_dict())
print("Share of the EU27 population weight in the affected countries: %.1f%%" % (100 * df.loc[df["affected"] == "Affected countries", "weight_eu27"].sum() / df["weight_eu27"].sum()))
partB.round(2)

In [ ]:
# The same pairs for every country: where does each country sit?
by_country_pairs = pd.DataFrame({c: {label: rho(d, a, b) for a, b, label in PAIRS[:8]} for c, d in df.groupby("country")}).T
by_country_pairs["Average of the eight"] = by_country_pairs.mean(axis=1)
by_country_pairs["Group"] = np.where(by_country_pairs.index.isin(AFFECTED_EU), "Affected", "Other 20")
by_country_pairs = by_country_pairs.sort_values("Average of the eight")
by_country_pairs.round(2)

In [ ]:
# The same thing in plain percentages
def pct(mask, col): return df[mask].groupby("affected")[col].mean().mul(100).round(1)
medium, micro = df["size"].str.startswith("Medium"), df["size"].str.startswith("Micro")
high_inv, no_inv = df["investment_level"] >= 3, df["investment_level"] == 0
partB2 = pd.DataFrame({
    "On-site %: medium firms": pct(medium, "on_site"), "On-site %: micro firms": pct(micro, "on_site"),
    "On-site %: invest 6% or more": pct(high_inv, "on_site"), "On-site %: invest nothing": pct(no_inv, "on_site"),
    "Purchased %: has climate strategy": pct(df["climate_strategy"] == "Has a strategy", "purchased"),
    "Purchased %: no plans for one": pct(df["climate_strategy"] == "No plans", "purchased"),
    "Plan renewables %: 5+ other actions": pct((df["pathway"] == "No renewable") & (df["n_other_actions"] >= 5), "plans_renewable"),
    "Plan renewables %: 1-2 other actions": pct((df["pathway"] == "No renewable") & (df["n_other_actions"].between(1, 2)), "plans_renewable")}).T
partB2[["Other 20 countries", "Affected countries"]]

**What to look for.** In the other 20 countries each pair is clearly related. In the affected countries every pair should be close to zero, except the control in the last row, which links two questions that sit next to each other in the questionnaire. That pattern, intact within a block of questions and broken between blocks, is what you would see if blocks of answers had been attached to the wrong firms within each country. It could also come from interviews that were not genuine. The file alone cannot tell us which.

## C. Can the models predict anything inside those countries?

For each country, how well does the full model rank that country's own firms? An AUC of 0.50 means no better than tossing a coin. Predictions come from 5-fold cross-validation with all 57 predictors.

In [ ]:
t0 = time.time()
B = SETS["B All"]
proba = np.zeros((len(df), 3))
for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(df, y):
    p = make_pre(B); Xa = p.fit_transform(df.iloc[tr][B]); Xb = p.transform(df.iloc[te][B])
    proba[te] = new_logit("B All").fit(Xa, y.iloc[tr]).predict_proba(Xb)
within = pd.DataFrame({"Within-country AUC": {c: roc_auc_score(y[df["country"] == c], proba[(df["country"] == c).values], multi_class="ovr") for c in sorted(df["country"].unique())}})
within["Firms"] = df["country"].value_counts(); within["Group"] = np.where(within.index.isin(AFFECTED_EU), "Affected", "Other 20")
within = within.sort_values("Within-country AUC")
partC = within.groupby("Group")["Within-country AUC"].agg(["min", "median", "max"]).round(3)
print(f"took {(time.time() - t0) / 60:.1f} minutes"); display(partC); within.round(3)

## D. The main results with and without those countries

Everything below is run on all 27 countries and on the 20 countries used in the paper (and, for prediction, on the affected seven alone). Model settings are the ones selected on Day 2.

In [ ]:
t0 = time.time()
# The country that passes the check with the lowest value is also left out in one extra run, to see whether the threshold matters.
passing = partA[(partA["EU27"] == "yes") & ~partA.index.isin(AFFECTED_EU)]
BORDERLINE = passing["Employees vs turnover"].idxmin()
print("Lowest passing EU country: %s (%.2f)" % (BORDERLINE, passing["Employees vs turnover"].min()))
SAMPLES = {"All 27 countries": df.index, "20 countries": df.index[~df["country"].isin(AFFECTED_EU)],
           f"19 countries (also without {BORDERLINE})": df.index[~df["country"].isin(AFFECTED_EU + [BORDERLINE])],
           "Affected countries only": df.index[df["country"].isin(AFFECTED_EU)]}
MAIN = "No survey weights (main)"
runs6, rows = {}, []
for name, idx in SAMPLES.items():
    for s in range(10):
        tr, te = train_test_split(idx, test_size=0.20, stratify=y.loc[idx], random_state=2000 + s)
        for set_name in SETS:
            res = run_split(tr, te, set_name, MAIN)
            rows.append({"Sample": name, "Predictors": set_name, **{m: auc(y.loc[te], p) for m, p in res["proba"].items()}})
            if set_name == "B All":
                del res["fitted"]; runs6[(name, MAIN, s)] = res
                if name in ("All 27 countries", "20 countries"):
                    r2 = run_split(tr, te, set_name, "Population weights"); del r2["fitted"]; runs6[(name, "Population weights", s)] = r2
d1 = pd.DataFrame(rows).groupby(["Sample", "Predictors"], sort=False)[["Logistic regression", "XGBoost"]].mean()
d1["XGBoost minus logistic"] = d1["XGBoost"] - d1["Logistic regression"]; d1 = d1.round(3)
print(f"took {(time.time() - t0) / 60:.1f} minutes"); print("Mean test AUC over 10 random splits"); d1

In [ ]:
THEMES = sorted(set(theme_of(p) for p in B))
def summary6(sample, scheme, model):
    P = pd.DataFrame({s: runs6[(sample, scheme, s)]["themes"][model]["Purchased"] for s in range(10)})
    O = pd.DataFrame({s: runs6[(sample, scheme, s)]["themes"][model]["On-site"] for s in range(10)})
    return pd.DataFrame({"Purchased": P.mean(axis=1), "On-site": O.mean(axis=1), "Ratio (median)": (O / P).median(axis=1), "Splits > 1": ((O / P) > 1).sum(axis=1)})
d2 = pd.concat({f"{sample} | {model}": summary6(sample, MAIN, model) for sample in list(SAMPLES)[:3] for model in ["XGBoost", "Logistic regression"]}, axis=1)
order = d2[("20 countries | XGBoost", "On-site")].sort_values(ascending=False).index
print("Theme importance and the on-site / purchased ratio, no survey weights. 'Splits > 1' is out of 10.")
d2.loc[order].round(3)

In [ ]:
d3 = pd.concat({f"{sample} | {model}": summary6(sample, "Population weights", model)[["Ratio (median)", "Splits > 1"]]
                for sample in ["All 27 countries", "20 countries"] for model in ["XGBoost", "Logistic regression"]}, axis=1)
print("The same ratio with Population weights"); d3.loc[order].round(2)

In [ ]:
# Descriptive results that do not need a model
def shares(frame, by):
    w = frame["weight_eu27"]
    return (100 * w.groupby([frame[by], frame["pathway"]]).sum().unstack().pipe(lambda t: t.div(t.sum(axis=1), axis=0)))[["No renewable", "Purchased", "On-site"]].round(1)
keep = ~df["country"].isin(AFFECTED_EU)
d4 = pd.concat({"All 27 countries": shares(df, "size"), "20 countries": shares(df[keep], "size"), "Affected countries only": shares(df[~keep], "size")}, axis=1)
tot = pd.DataFrame({name: (100 * frame.groupby("pathway")["weight_eu27"].sum() / frame["weight_eu27"].sum()).round(1)
                    for name, frame in [("All 27 countries", df), ("20 countries", df[keep]), ("Affected countries only", df[~keep])]}).T[["No renewable", "Purchased", "On-site"]]
print("Weighted pathway shares (%)"); display(tot); print("By firm size (weighted %)"); d4

In [ ]:
# Country-level result: purchasing and national income
w = df["weight_country"]
raw_share = (w.groupby([df["country"], df["pathway"]]).sum().unstack().pipe(lambda t: t.div(t.sum(axis=1), axis=0)) * 100)[["Purchased", "On-site"]].round(1)
cty = raw_share.join(indicators)
rows = []
for name, sub in [("All 27 countries", cty), ("20 countries", cty[~cty.index.isin(AFFECTED_EU)])]:
    for out in ["Purchased", "On-site"]:
        for ind, label in [("gdp_per_capita_pps_2023_eu27_100", "Income per person"), ("renewable_share_electricity_2023_pct", "Renewable share of electricity"),
                           ("electricity_price_business_2023S2_eur_kwh", "Business electricity price")]:
            rho_, p_ = spearmanr(sub[out], sub[ind]); rows.append({"Countries": name, "Pathway share": out, "Indicator": label, "rho": round(rho_, 2), "p": round(p_, 3), "n": len(sub)})
d5 = pd.DataFrame(rows)
print("Affected countries: income index and purchasing share"); display(cty.loc[AFFECTED_EU, ["Purchased", "On-site", "gdp_per_capita_pps_2023_eu27_100"]])
d5

In [ ]:
# Planning among non-users
non = df[df["pathway"] == "No renewable"]
bands = pd.cut(non["n_other_actions"], [-1, 0, 2, 4, 6, 8], labels=["none", "1 or 2", "3 or 4", "5 or 6", "7 or 8"])
d6 = non.assign(band=bands).pivot_table(index="band", columns="affected", values="plans_renewable", aggfunc=lambda s: round(100 * s.mean(), 1), observed=True)
d6.loc["All non-users"] = non.groupby("affected")["plans_renewable"].mean().mul(100).round(1)
d6.loc["Firms"] = non["affected"].value_counts()
print("Share of non-users planning to adopt (%), by number of other actions"); d6[["Other 20 countries", "Affected countries"]]

## Save the outputs

In [ ]:
flat = lambda t: t.set_axis([" | ".join(map(str, c)) for c in t.columns], axis=1)
with pd.ExcelWriter("Day6_results.xlsx") as xl:
    partA.round(6).to_excel(xl, sheet_name="A employees vs turnover")
    partB.round(6).to_excel(xl, sheet_name="B pairs of answers")
    partB2.to_excel(xl, sheet_name="B in percentages")
    by_country_pairs.round(3).to_excel(xl, sheet_name="B by country")
    q13_rate = lambda g: 100 * df[(df["q1_3"] == 1) & (df["affected"] == g)].groupby("country")["uses_renewable"].mean()
    pd.DataFrame({"Item": ["Firms in affected EU countries", "Firms in the other 20", "Share of EU27 population weight in affected countries %", "Lowest passing EU country", "Its employees-turnover correlation",
                           "Q1.3 mentioners on a renewable pathway %, affected: lowest country", "same: highest", "other 20: lowest country", "same: highest"],
                  "Value": [int((df["affected"] == "Affected countries").sum()), int((df["affected"] != "Affected countries").sum()),
                            round(100 * df.loc[df["affected"] == "Affected countries", "weight_eu27"].sum() / df["weight_eu27"].sum(), 1), BORDERLINE, round(passing["Employees vs turnover"].min(), 3),
                            round(q13_rate("Affected countries").min(), 1), round(q13_rate("Affected countries").max(), 1), round(q13_rate("Other 20 countries").min(), 1), round(q13_rate("Other 20 countries").max(), 1)]}).to_excel(xl, sheet_name="Notes", index=False)
    within.round(6).to_excel(xl, sheet_name="C within-country AUC")
    d1.reset_index().to_excel(xl, sheet_name="D AUC", index=False)
    flat(d2.loc[order].round(6)).to_excel(xl, sheet_name="D themes unweighted")
    flat(d3.loc[order].round(3)).to_excel(xl, sheet_name="D ratios EU27 weights")
    tot.to_excel(xl, sheet_name="D pathway shares")
    flat(d4).to_excel(xl, sheet_name="D shares by size")
    d5.to_excel(xl, sheet_name="D country correlations", index=False)
    d6.to_excel(xl, sheet_name="D planning")
print("Saved Day6_results.xlsx")
try:
    from google.colab import files; files.download("Day6_results.xlsx")
except ImportError:
    pass